# 14. 9차 사전 등록 (김유환): 베이스라인 + UME 자리 (s4 확증, 학습 없음)

`configs/prereg_yh_slots.yaml` 대로 실행한다. s4의 B2 계산(약 1분) → 결합 · 분석(약 10분). 7 · 8차의 저장 벡터를 쓴다.

In [ ]:
# @title 0. 공통 준비 (모든 노트북 동일)
import sys
if any(m.startswith("msrs_paper") for m in sys.modules):
    raise RuntimeError("이전에 불러온 코드가 메모리에 남아 있습니다. 메뉴 '런타임 → 세션 다시 시작' 후 이 셀부터 다시 실행하세요.")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

REPO_URL = "https://github.com/yuhwani/multimodal-recsys.git"  # upstream에 머지되면 3sunghynix 주소로 변경
BRANCH = "main"
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -r /content/repo/paper/requirements.txt
!cd /content/repo && git log -1 --format="코드 커밋: %h %s (%cd)"

import sys
sys.path.insert(0, "/content/repo/paper")
from msrs_paper.config import load_all, Paths
cfg, exps, plan = load_all()
DRIVE_ROOT = cfg["drive_root"]  # 공유 폴더 바로가기 경로가 다르면 이 줄만 바꾼다
paths = Paths(DRIVE_ROOT).make_dirs()
print("드라이브 폴더:", paths.root)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title 1. 준비물 · 사전 등록 확인 + s4 베이스라인 B2 계산
from msrs_paper.utils import verify_manifest
from msrs_paper.config import load_yaml
from msrs_paper import baselines
m = verify_manifest(paths)
prereg = load_yaml("prereg_yh_slots")
assert prereg["status"] == "final"
baselines.run(cfg, paths, "B2", "s4", group="YH7_BASE")
need = [paths.run_dir("YH5_UME", "s4", f"{c}_seed{s}") / "user_vectors.npy" for c in ("C0", "C1") for s in (42, 43, 44)]
print(f"준비물 {len(m['files'])}개 일치 · s4 UME 벡터 {sum(f.exists() for f in need)} / 6")

In [ ]:
# @title 2. 분석: 확증 H18 · H19 + 자리 수 · 분할별 탐색 (약 10분)
from msrs_paper import analysis_yh7
confirmatory, table, contrasts = analysis_yh7.run(cfg, exps, paths)
table[table["segment"] == "all"].pivot_table(index="series", columns="split", values="R").round(4)

In [ ]:
# @title 3. 탐색: 전용 자리 대비 · 구간별
c = contrasts[contrasts["description"].str.contains(" 20")]
display(c[c["segment"] == "all"][["split", "description", "estimate", "ci_low", "ci_high"]])
display(c[c["segment"].isin(["established", "new_item", "sparse"])].pivot_table(
    index=["split", "description"], columns="segment", values="estimate").round(4))